# Setup
Connect your existing folder with the pre-collected experiments data.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import ast
import numpy as np
import matplotlib.pyplot as plt

DRIVE_ROOT = Path("/content/drive/MyDrive/EM43")
print("DRIVE_ROOT =", DRIVE_ROOT.resolve())

In [ ]:
def lut_idx4(l: int, c: int, r: int) -> int:
    return (l * 4 + c) * 4 + r


IDX_000 = lut_idx4(0, 0, 0)
IDX_020 = lut_idx4(0, 2, 0)
IDX_200 = lut_idx4(2, 0, 0)
IDX_002 = lut_idx4(0, 0, 2)


def enforce_immutables_full(rule_full: np.ndarray) -> None:
    if rule_full.shape[-1] != 64:
        raise ValueError(f"rule_full last dimension must be 64, got {rule_full.shape}")
    rule_full[..., IDX_000] = 0
    rule_full[..., IDX_020] = 2
    rule_full[..., IDX_200] = 0
    rule_full[..., IDX_002] = 0


def parse_uint8_vector_string(s: str, expected_len: int | None = None, name: str = "vector") -> np.ndarray:
    s = s.strip()
    if not s:
        raise ValueError(f"{name} string is empty")

    if s[0] != "[":
        s = "[" + s + "]"

    try:
        obj = ast.literal_eval(s)
    except Exception as e:
        raise ValueError(f"Could not parse {name} string: {e}") from e

    arr = np.asarray(obj, dtype=np.int64).reshape(-1)

    if expected_len is not None and arr.shape[0] != expected_len:
        raise ValueError(f"{name} must have length {expected_len}, got {arr.shape[0]}")
    if np.any(arr < 0) or np.any(arr > 3):
        raise ValueError(f"{name} values must be in {{0,1,2,3}}")

    return arr.astype(np.uint8)


def parse_rule_program_strings(rule_str: str, prog_str: str):
    rule = parse_uint8_vector_string(rule_str, expected_len=64, name="rule")
    prog = parse_uint8_vector_string(prog_str, expected_len=None, name="program")
    enforce_immutables_full(rule)
    return rule, prog


def load_genome_from_drive(task_alias: str, run_id: int, drive_root: Path = DRIVE_ROOT):
    task_dir = drive_root / task_alias
    genome_path = task_dir / f"best_em43_{task_alias}_run_{run_id:03d}.npz"

    if not genome_path.exists():
        raise FileNotFoundError(f"Genome file not found: {genome_path}")

    data = np.load(genome_path, allow_pickle=True)

    rule_full = np.asarray(data["rule_full"], dtype=np.uint8).reshape(-1)
    program = np.asarray(data["program"], dtype=np.uint8).reshape(-1)

    if rule_full.shape != (64,):
        raise ValueError(f"Loaded rule has wrong shape: {rule_full.shape}")

    enforce_immutables_full(rule_full)

    meta = {
        "path": genome_path,
        "task_alias": str(data["task_alias"]) if "task_alias" in data else task_alias,
        "run": int(data["run"]) if "run" in data else run_id,
        "Lp": int(data["Lp"]) if "Lp" in data else int(program.shape[0]),
        "best_gen": int(data["best_gen"]) if "best_gen" in data else None,
        "fitness": float(data["fitness"]) if "fitness" in data else None,
        "mae": float(data["mae"]) if "mae" in data else None,
    }
    return rule_full, program, meta


def build_single_input_task(*, inputs, target_fn):
    xs = list(inputs)
    encoded_inputs = []
    targets = np.empty((len(xs),), dtype=np.int64)

    for i, x in enumerate(xs):
        if x < 0:
            raise ValueError(f"Input x={x} must be nonnegative")

        enc = np.zeros((x + 1,), dtype=np.uint8)
        enc[x] = 2
        encoded_inputs.append(enc)
        targets[i] = int(target_fn(x))

    return encoded_inputs, targets


def build_two_input_task(*, pairs, target_fn):
    pairs = list(pairs)
    encoded_inputs = []
    targets = np.empty((len(pairs),), dtype=np.int64)

    for i, (a, b) in enumerate(pairs):
        if a < 0 or b < 0:
            raise ValueError(f"Inputs must be nonnegative, got {(a, b)}")

        L = max(a, b) + 1
        enc = np.zeros((L,), dtype=np.uint8)
        enc[a] = 2
        enc[b] = 2
        encoded_inputs.append(enc)
        targets[i] = int(target_fn(a, b))

    return encoded_inputs, targets


class EM43NumpyEvaluator:
    """
    Pure-numpy EM43 evaluator.

    Matches training semantics:
      - same tape layout: [program][separator][encoding]
      - same LUT indexing
      - same synchronous radius-1 update
      - same halting rule: live > 0 and 2*ctrl >= live
      - same time-0 halting check
      - same immutables
      - if live == 0, continue until T_max

    Inference difference:
      - no right-boundary invalidation
      - initial usable tape length is:
            Lp + Ls + (max active input position + 1) + right_padding
      - right-only self-extension before step
      - decode over the whole usable right region from the fixed encoding origin
      - no left extension
    """

    def __init__(
        self,
        *,
        Lp: int,
        Ls: int = 4,
        T_max: int = 10000,
        right_padding: int = 20,
        extend_margin: int = 3,
        extend_by: int = 20,
    ):
        self.Lp = int(Lp)
        self.Ls = int(Ls)
        self.T_max = int(T_max)
        self.right_padding = int(right_padding)
        self.extend_margin = int(extend_margin)
        self.extend_by = int(extend_by)

        if self.Lp <= 0:
            raise ValueError("Lp must be positive")
        if self.Ls < 0:
            raise ValueError("Ls must be nonnegative")
        if self.T_max <= 0:
            raise ValueError("T_max must be positive")
        if self.right_padding < 0:
            raise ValueError("right_padding must be nonnegative")
        if self.extend_margin < 0:
            raise ValueError("extend_margin must be nonnegative")
        if self.extend_by <= 0:
            raise ValueError("extend_by must be positive")

        self.enc_origin = self.Lp + self.Ls

    def _validate_inputs(self, rule_full: np.ndarray, program: np.ndarray, enc: np.ndarray):
        if rule_full.shape != (64,):
            raise ValueError(f"rule_full shape must be (64,), got {rule_full.shape}")
        if program.ndim != 1 or program.shape[0] != self.Lp:
            raise ValueError(f"program shape must be ({self.Lp},), got {program.shape}")
        if enc.ndim != 1:
            raise ValueError(f"encoding must be 1D, got shape {enc.shape}")
        if enc.shape[0] <= 0:
            raise ValueError("encoding must be non-empty")

        if np.any(rule_full < 0) or np.any(rule_full > 3):
            raise ValueError("rule values must be in {0,1,2,3}")
        if np.any(program < 0) or np.any(program > 3):
            raise ValueError("program values must be in {0,1,2,3}")
        if np.any(enc < 0) or np.any(enc > 3):
            raise ValueError("encoding values must be in {0,1,2,3}")

    def _build_initial_tape(self, program: np.ndarray, enc: np.ndarray) -> np.ndarray:
        active = np.flatnonzero(enc != 0)
        if active.size == 0:
            enc_len_used = enc.shape[0]
        else:
            enc_len_used = int(active[-1]) + 1

        usable_len = self.Lp + self.Ls + enc_len_used + self.right_padding

        tape = np.zeros((usable_len,), dtype=np.uint8)
        tape[:self.Lp] = program
        tape[self.enc_origin:self.enc_origin + enc.shape[0]] = enc
        return tape

    @staticmethod
    def _count_live_ctrl(tape: np.ndarray):
        live = int(np.count_nonzero(tape))
        ctrl = int(np.count_nonzero(tape == 3))
        return live, ctrl

    @staticmethod
    def _rightmost_active_index(tape: np.ndarray):
        pos = np.flatnonzero(tape != 0)
        return None if pos.size == 0 else int(pos[-1])

    def _extend_if_needed(self, tape: np.ndarray) -> np.ndarray:
        r_active = self._rightmost_active_index(tape)
        if r_active is None:
            return tape

        right_edge = tape.shape[0] - 1
        if right_edge - r_active <= self.extend_margin:
            tape = np.concatenate([tape, np.zeros((self.extend_by,), dtype=np.uint8)])
        return tape

    @staticmethod
    def _step_once(rule_full: np.ndarray, tape: np.ndarray) -> np.ndarray:
        L = np.empty_like(tape)
        C = tape
        R = np.empty_like(tape)

        L[0] = 0
        L[1:] = tape[:-1]

        R[-1] = 0
        R[:-1] = tape[1:]

        idx = (L.astype(np.int32) * 4 + C.astype(np.int32)) * 4 + R.astype(np.int32)
        return rule_full[idx].astype(np.uint8, copy=False)

    def _decode(self, tape: np.ndarray):
        decode_region = tape[self.enc_origin:]
        pos = np.flatnonzero(decode_region == 2)
        return None if pos.size == 0 else int(pos[-1])

    def run_single(self, *, rule_full: np.ndarray, program: np.ndarray, enc: np.ndarray, return_trajectory: bool = False):
        rule = np.asarray(rule_full, dtype=np.uint8).copy()
        prog = np.asarray(program, dtype=np.uint8).copy()
        enc = np.asarray(enc, dtype=np.uint8).copy()

        enforce_immutables_full(rule)
        self._validate_inputs(rule, prog, enc)

        tape = self._build_initial_tape(prog, enc)
        n_extensions = 0
        trajectory = [tape.copy()] if return_trajectory else None

        live, ctrl = self._count_live_ctrl(tape)
        if live > 0 and ctrl * 2 >= live:
            pred = self._decode(tape)
            return {
                "pred": pred,
                "valid": pred is not None,
                "halted": True,
                "halt_time": 0,
                "n_extensions": n_extensions,
                "final_usable_len": int(tape.shape[0]),
                "trajectory": trajectory,
            }

        for t in range(1, self.T_max + 1):
            old_len = tape.shape[0]
            tape = self._extend_if_needed(tape)
            if tape.shape[0] != old_len:
                n_extensions += 1

            tape = self._step_once(rule, tape)

            if return_trajectory:
                trajectory.append(tape.copy())

            live, ctrl = self._count_live_ctrl(tape)
            if live > 0 and ctrl * 2 >= live:
                pred = self._decode(tape)
                return {
                    "pred": pred,
                    "valid": pred is not None,
                    "halted": True,
                    "halt_time": t,
                    "n_extensions": n_extensions,
                    "final_usable_len": int(tape.shape[0]),
                    "trajectory": trajectory,
                }

        return {
            "pred": None,
            "valid": False,
            "halted": False,
            "halt_time": None,
            "n_extensions": n_extensions,
            "final_usable_len": int(tape.shape[0]),
            "trajectory": trajectory,
        }

    def plot_trajectory(self, trajectory, *, title="EM43 trajectory", figsize=(14, 6), tick_step=10):
        if trajectory is None or len(trajectory) == 0:
            raise ValueError("trajectory is empty")

        max_len = max(row.shape[0] for row in trajectory)
        img = np.zeros((len(trajectory), max_len), dtype=np.uint8)
        for i, row in enumerate(trajectory):
            img[i, :row.shape[0]] = row

        from matplotlib.colors import ListedColormap, BoundaryNorm

        cmap = ListedColormap(["white", "black", "red", "blue"])
        norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(
            img,
            cmap=cmap,
            norm=norm,
            interpolation="nearest",
            aspect="equal",   # square cells
            origin="upper",
        )

        # Left boundary before cell 0
        ax.axvline(x=-0.5, color="gray", linewidth=2)

        # Encoding origin boundary x0 = enc_origin
        ax.axvline(x=self.enc_origin - 0.5, color="green", linewidth=2)

        ax.set_xlabel("Encoding position (relative to x0)")
        ax.set_ylabel("Time step")
        ax.set_title(title)

        ax.set_xlim(-0.5, max_len - 0.5)
        ax.set_ylim(len(trajectory) - 0.5, -0.5)

        # X ticks labeled relative to encoding origin
        tick_start = self.enc_origin
        ticks = np.arange(tick_start, max_len, tick_step)
        ax.set_xticks(ticks)
        ax.set_xticklabels((ticks - self.enc_origin).astype(int))

        plt.show()

# Run validation on the desired range
1. to validate a trained em on a task: use the same alias you used for training
2. select a run id
3. set hyperparams
4. set task formula
5. watch output

In [ ]:
# ---- Genome source: file (default) ----
task_alias = "mod2c"
run_id = 2

rule_full, program, meta = load_genome_from_drive(task_alias=task_alias, run_id=run_id)
print("Loaded genome from:", meta["path"])
print("Loaded task_alias  :", meta["task_alias"])
print("Loaded run         :", meta["run"])
print("Loaded Lp          :", meta["Lp"])
print("Loaded best_gen    :", meta["best_gen"])
print("Loaded fitness     :", meta["fitness"])
print("Loaded mae         :", meta["mae"])

# ---- Genome source: manual (alternative) ----
#rule_str ="[0, 0, 0, 3, 1, 1, 3, 2, 2, 1, 2, 3, 2, 0, 3, 1, 1, 0, 3, 3, 1, 0, 0, 3, 3, 1, 3, 3, 1, 2, 2, 1, 0, 3, 1, 1, 2, 3, 1, 3, 2, 1, 1, 3, 3, 2, 2, 2, 3, 0, 3, 3, 0, 3, 1, 3, 2, 2, 3, 3, 1, 3, 2, 3]"

#prog_str = "[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 2, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1]"

#rule_full, program = parse_rule_program_strings(rule_str, prog_str)
#print("Loaded genome manually.")

Lp = int(program.shape[0])

evaluator = EM43NumpyEvaluator(
    Lp=Lp,
    Ls=4,
    T_max=4000,
    right_padding=20,
    extend_margin=3,
    extend_by=20,
)

# ---- Build dataset ----
# 1-input default:
encoded_inputs, targets = build_single_input_task(
    inputs=range(0, 1000),
    target_fn=lambda x: x*(x%2+1),   # edit here
)

# 2-input alternative:
# pairs = [(a, b) for a in range(0, 8) for b in range(0, 8)]
# encoded_inputs, targets = build_two_input_task(
#     pairs=pairs,
#     target_fn=lambda a, b: a + b,
# )

# ---- Evaluate dataset ----
M = len(encoded_inputs)
preds = np.full((M,), -1, dtype=np.int64)
valid = np.zeros((M,), dtype=bool)
halt_times = np.full((M,), -1, dtype=np.int64)
n_extensions = np.zeros((M,), dtype=np.int64)
final_lengths = np.zeros((M,), dtype=np.int64)

for i in range(M):
    out = evaluator.run_single(
        rule_full=rule_full,
        program=program,
        enc=encoded_inputs[i],
        return_trajectory=False,
    )
    preds[i] = -1 if out["pred"] is None else int(out["pred"])
    valid[i] = bool(out["valid"])
    halt_times[i] = -1 if out["halt_time"] is None else int(out["halt_time"])
    n_extensions[i] = int(out["n_extensions"])
    final_lengths[i] = int(out["final_usable_len"])

correct = valid & (preds == targets)
accuracy = float(correct.mean())
valid_fraction = float(valid.mean())
all_correct = bool(np.all(correct))

print()
print("Dataset result")
print("  n_examples            =", M)
print("  all_correct           =", all_correct)
print("  exact_match_accuracy  =", accuracy)
print("  valid_fraction        =", valid_fraction)
print("  n_wrong               =", int((~correct).sum()))

# Visualize the space-time plot of a CA solving a task
1. select a particular input
2. set the task (if you want to see predicted vs actual value)

In [ ]:
# Choose one x directly
x = 35
target_fn = lambda x: x*(x%2+1)   # keep aligned with Cell 3 if you want direct comparison

enc = np.zeros((x + 1,), dtype=np.uint8)
enc[x] = 2
target = int(target_fn(x))

out = evaluator.run_single(
    rule_full=rule_full,
    program=program,
    enc=enc,
    return_trajectory=True,
)

print("Single-example result")
print("  x                 =", x)
print("  pred              =", out["pred"])
print("  target            =", target)
print("  correct           =", bool(out["valid"] and out["pred"] == target))
print("  valid             =", out["valid"])
print("  halted            =", out["halted"])
print("  halt_time         =", out["halt_time"])
print("  n_extensions      =", out["n_extensions"])
print("  final_usable_len  =", out["final_usable_len"])

evaluator.plot_trajectory(
    out["trajectory"],
    title=f"EM43 trajectory (x={x}, target={target})",
)

# Evaluate all saved runs for a task

In [ ]:
task_alias = "rounddiv3"   # edit here

# Keep aligned with Cell 3 task definition
target_fn = lambda x: round(x/3)   # edit here

# Keep aligned with Cell 3 dataset definition
encoded_inputs, targets = build_single_input_task(
    inputs=range(0, 1000),
    target_fn=target_fn,
)

task_dir = DRIVE_ROOT / task_alias
if not task_dir.exists():
    raise FileNotFoundError(f"Task directory not found: {task_dir}")

genome_paths = sorted(task_dir.glob(f"best_em43_{task_alias}_run_*.npz"))
if len(genome_paths) == 0:
    raise FileNotFoundError(f"No genomes found in {task_dir}")

rows = []

print(f"Found {len(genome_paths)} genomes for task '{task_alias}'")
print()

for genome_path in genome_paths:
    data = np.load(genome_path, allow_pickle=True)

    rule_full = np.asarray(data["rule_full"], dtype=np.uint8).reshape(-1)
    program = np.asarray(data["program"], dtype=np.uint8).reshape(-1)
    enforce_immutables_full(rule_full)

    run_id = int(data["run"]) if "run" in data else int(genome_path.stem.split("_")[-1])
    saved_best_gen = int(data["best_gen"]) if "best_gen" in data else None
    saved_fitness = float(data["fitness"]) if "fitness" in data else None
    saved_mae = float(data["mae"]) if "mae" in data else None

    evaluator = EM43NumpyEvaluator(
        Lp=len(program),
        Ls=4,
        T_max=10000,
        right_padding=20,
        extend_margin=3,
        extend_by=20,
    )

    M = len(encoded_inputs)
    preds = np.full((M,), -1, dtype=np.int64)
    valid = np.zeros((M,), dtype=bool)

    for i in range(M):
        out = evaluator.run_single(
            rule_full=rule_full,
            program=program,
            enc=encoded_inputs[i],
            return_trajectory=False,
        )
        preds[i] = -1 if out["pred"] is None else int(out["pred"])
        valid[i] = bool(out["valid"])

    correct = valid & (preds == targets)
    accuracy = float(correct.mean())
    valid_fraction = float(valid.mean())
    all_correct = bool(np.all(correct))
    n_wrong = int((~correct).sum())

    rows.append([
        run_id,
        len(program),
        saved_best_gen,
        saved_fitness,
        saved_mae,
        accuracy,
        valid_fraction,
        all_correct,
        n_wrong,
        genome_path.name,
    ])

rows.sort(key=lambda r: r[0])

print("Results by run")
print("run | Lp | best_gen | saved_fit | saved_mae | acc | valid_frac | all_correct | n_wrong | file")
for r in rows:
    print(
        f"{r[0]:>3} | {r[1]:>2} | {str(r[2]):>8} | "
        f"{str(None if r[3] is None else round(r[3], 6)):>9} | "
        f"{str(None if r[4] is None else round(r[4], 6)):>9} | "
        f"{r[5]:.6f} | {r[6]:.6f} | {str(r[7]):>11} | {r[8]:>7} | {r[9]}"
    )

best_idx = int(np.argmax([r[5] for r in rows]))
best_row = rows[best_idx]

print()
print("Best run on this evaluation set")
print("  run              =", best_row[0])
print("  accuracy         =", best_row[5])
print("  valid_fraction   =", best_row[6])
print("  all_correct      =", best_row[7])
print("  n_wrong          =", best_row[8])
print("  genome_file      =", best_row[9])